# Chirp Evaluation Manifest Generator

This notebook prepares Chirp's transcription manifest for evaluation. It merges raw transcription results with ground-truth segmentations.

**Runtime:** designed to run in a Jupyter kernel whose CWD is `model/colabs/` (so the sibling `common/` package is importable). The `notebook_docker` compose service at `model/notebook_docker/` provides that, with the required `loguru` and `google-cloud-storage` already installed. A stock `Open in Colab` badge would land in a kernel without either, so the badge was removed.

**Note:** This version is focused strictly on data merging and manifest generation. Analysis (WER calculation) and visualization are handled in a separate benchmark notebook.

It performs the following steps:

1.  **Loads existing ground truth** from the pilot manifest (`{PROJECT_NAME}_transcriptions.json`).
2.  **Maps Chirp segments** using the `batch_manifest.jsonl` to align raw API results with the benchmark offsets.
3.  **Generates a merged manifest** specifically containing the `pred_text_chirp_v3` field.
4.  **Exports the final benchmark** to Google Cloud Storage.

In [ ]:
import collections
import json
import re
from pathlib import Path
from typing import Any

from google.cloud import storage
from google.colab import auth
from loguru import logger

# @markdown ### GCP Configuration
GCP_PROJECT_ID = ""  # @param {type:"string"}
GCS_BUCKET = ""  # @param {type:"string"}
PROJECT_NAME = ""  # @param {type:"string"}
EXPERIMENT_NAME = ""  # @param {type:"string"}
MODEL_ID = "chirp_3"  # @param ["chirp_3", "chirp_telephony"] {type:"string"}

assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided and cannot be empty."
assert GCS_BUCKET, "GCS_BUCKET must be provided and cannot be empty."
assert PROJECT_NAME, "PROJECT_NAME must be provided and cannot be empty."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided and cannot be empty."

# Input/Output Paths
GCS_SEGMENT_MAP_DIR = f"segmented_audio/{PROJECT_NAME}_audio"
GCS_INPUT_BENCHMARK_PATH = f"manifests/{PROJECT_NAME}_transcriptions.json"
GCS_OUTPUT_MANIFEST_DIR = f"inference_manifests/{MODEL_ID}"
GCS_RAW_TRANSCRIPTS_DIR = (
    f"transcripts/{PROJECT_NAME}_audio/{MODEL_ID}/{EXPERIMENT_NAME}"
)

# Filenames
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"
EXISTING_BENCHMARK_FILENAME = f"{PROJECT_NAME}_transcriptions.json"
UPDATED_BENCHMARK_FILENAME = f"{EXPERIMENT_NAME}.jsonl"

# Authenticate to GCS
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT_ID} --quiet

from common.manifest import load_manifest

In [ ]:
# @title Defining helper functions


def merge_gcs_results_to_manifest(
    baseline_data: list[dict[str, Any]],
    batch_manifest_data: list[dict[str, Any]],
    gcs_bucket_name: str,
    output_file: str,
) -> list[dict[str, Any]]:
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(gcs_bucket_name)

    logger.info(
        f"Fetching Chirp transcripts from gs://{gcs_bucket_name}/{GCS_RAW_TRANSCRIPTS_DIR}/..."
    )
    transcript_blobs = list(
        bucket.list_blobs(prefix=f"{GCS_RAW_TRANSCRIPTS_DIR}/")
    )

    chirp_predictions = {}
    for blob in transcript_blobs:
        if not blob.name.endswith(".json"):
            continue

        match = re.search(r"([^/]+)__seg(\d{3})", Path(blob.name).name)
        if not match:
            continue

        example_id = match.group(1)
        seg_key = match.group(2)

        data = json.loads(blob.download_as_text())
        transcript_parts = []
        for res in data.get("results", []):
            alts = res.get("alternatives", [])
            if alts:
                top_alt = alts[0]
                if "transcript" in top_alt:
                    transcript_parts.append(top_alt["transcript"])

        chirp_predictions[(example_id, seg_key)] = " ".join(
            transcript_parts
        ).strip()

    offset_to_seg = collections.defaultdict(dict)
    for entry in batch_manifest_data:
        e_id = entry.get("example_id", "")
        s_id = entry.get("segment_id", "")
        offset = float(entry.get("offset", 0.0))
        offset_to_seg[e_id][s_id] = offset

    merged_records = []
    unmatched_count = 0

    for b_info in baseline_data:
        example_id = Path(b_info["audio_filepath"]).stem
        b_offset = float(b_info.get("offset", 0.0))

        matched_seg_id = None
        if example_id in offset_to_seg:
            for s_id, m_offset in offset_to_seg[example_id].items():
                if abs(b_offset - m_offset) < 0.25:
                    matched_seg_id = s_id
                    break

        chirp_text = ""
        if matched_seg_id and (example_id, matched_seg_id) in chirp_predictions:
            chirp_text = chirp_predictions[(example_id, matched_seg_id)]
        else:
            unmatched_count += 1

        record = {
            "audio_filepath": b_info["audio_filepath"],
            "text": b_info.get("text", ""),
            f"pred_text_{MODEL_ID}": chirp_text,
            "duration": b_info.get("duration", 0.0),
            "offset": b_offset,
            "lang": b_info.get("lang", "en"),
        }
        merged_records.append(record)

    if unmatched_count > 0:
        logger.warning(
            f"{unmatched_count} rows failed to map to a Chirp prediction."
        )

    with open(output_file, "w", encoding="utf-8") as f_out:
        f_out.writelines(json.dumps(rec) + "\n" for rec in merged_records)

    logger.info(
        f"Successfully wrote {len(merged_records)} merged records to {output_file}"
    )
    return merged_records


def run_pipeline() -> None:
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)

    bucket.blob(GCS_INPUT_BENCHMARK_PATH).download_to_filename(
        EXISTING_BENCHMARK_FILENAME
    )
    bucket.blob(
        f"{GCS_SEGMENT_MAP_DIR}/{BATCH_MANIFEST_FILENAME}"
    ).download_to_filename(BATCH_MANIFEST_FILENAME)

    baseline_data = load_manifest(EXISTING_BENCHMARK_FILENAME)
    batch_manifest_data = load_manifest(BATCH_MANIFEST_FILENAME)

    merge_gcs_results_to_manifest(
        baseline_data,
        batch_manifest_data,
        GCS_BUCKET,
        UPDATED_BENCHMARK_FILENAME,
    )

    bucket.blob(
        f"{GCS_OUTPUT_MANIFEST_DIR}/{UPDATED_BENCHMARK_FILENAME}"
    ).upload_from_filename(UPDATED_BENCHMARK_FILENAME)
    logger.info("Uploaded final manifest to GCS.")

In [ ]:
# @title Run Processing
run_pipeline()